Cell 1 — Import và đọc dữ liệu

In [1]:
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

df = pd.read_csv("../data/processed/flight_data_7m_selected.csv")

print("Dataset:", df.shape)

Dataset: (6965266, 17)


Cell 2 — Tách Feature và Target

In [2]:
FEATURE_COLUMNS = [
    "month",
    "day_of_month",
    "day_of_week",
    "fl_date",
    "op_unique_carrier",
    "op_carrier_fl_num",
    "origin",
    "dest",
    "crs_dep_time",
    "crs_arr_time",
    "crs_elapsed_time",
    "distance"
]

TARGET_COLUMNS = [
    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay"
]

X = df[FEATURE_COLUMNS].copy()

# Chuyển target thành multi-label 0/1
Y = (df[TARGET_COLUMNS] > 0).astype(int)

print("X:", X.shape)
print("Y:", Y.shape)

X: (6965266, 12)
Y: (6965266, 5)


Cell 3 — Chia Train 70% / Validation 15% / Test 15%

In [3]:
# Chia Train 70% và phần còn lại 30%
split_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(split_1.split(X, Y))

X_train = X.iloc[train_idx]
Y_train = Y.iloc[train_idx]

X_temp = X.iloc[temp_idx]
Y_temp = Y.iloc[temp_idx]


# Chia 30% còn lại thành Validation 15% và Test 15%
split_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(split_2.split(X_temp, Y_temp))

X_val = X_temp.iloc[val_idx]
Y_val = Y_temp.iloc[val_idx]

X_test = X_temp.iloc[test_idx]
Y_test = Y_temp.iloc[test_idx]

Cell 4 — Kiểm tra kết quả

In [4]:
print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

print("\nTỷ lệ:")
print(f"Train      : {len(X_train) / len(df):.2%}")
print(f"Validation : {len(X_val) / len(df):.2%}")
print(f"Test       : {len(X_test) / len(df):.2%}")

Train      : (4875686, 12)
Validation : (1044790, 12)
Test       : (1044790, 12)

Tỷ lệ:
Train      : 70.00%
Validation : 15.00%
Test       : 15.00%


Cell 5 — Kiểm tra phân bố 5 target

In [5]:
distribution = pd.DataFrame({
    "Train": Y_train.mean(),
    "Validation": Y_val.mean(),
    "Test": Y_test.mean()
}) * 100

distribution.round(2)

,Train,Validation,Test
carrier_delay,11.33,11.33,11.33
weather_delay,1.28,1.28,1.28
nas_delay,10.42,10.42,10.42
security_delay,0.11,0.11,0.11
late_aircraft_delay,10.67,10.67,10.67


Tạo thư mục và LƯU 3 FILE

In [6]:
import pandas as pd
from pathlib import Path
OUTPUT_PATH = Path("../data/processed/split/7m")

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

train_df = pd.concat(
    [X_train, Y_train],
    axis=1
)

val_df = pd.concat(
    [X_val, Y_val],
    axis=1
)

test_df = pd.concat(
    [X_test, Y_test],
    axis=1
)

train_df.to_csv(
    OUTPUT_PATH / "train.csv",
    index=False,
    encoding="utf-8-sig"
)

val_df.to_csv(
    OUTPUT_PATH / "validation.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    OUTPUT_PATH / "test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:")

print(OUTPUT_PATH / "train.csv")
print(OUTPUT_PATH / "validation.csv")
print(OUTPUT_PATH / "test.csv")

Đã lưu:
..\data\processed\split\7m\train.csv
..\data\processed\split\7m\validation.csv
..\data\processed\split\7m\test.csv
